# AllPower: six-hour energy preparation

**Status:** the bundled demand and hospital cases are simulated. This notebook checks the scenario, evaluates a seasonal baseline, and shows how to train the direct ML forecasters after a verified official CSV is added. It does not turn scenario outcomes into real-world outage probabilities. Run from the repository root.

In [ ]:
import json, platform, sys
from pathlib import Path
print({"python": sys.version.split()[0], "architecture": platform.machine(), "cwd": str(Path.cwd())})


## 1. Inspect provenance and assumptions

The fictional hospital uses 420 kW of critical demand, a 900 kWh battery with 720 kWh initially available, a 330 kW generator, and a fuel budget equivalent to 1,800 kWh. Regional supply and reserve are scenario assumptions.

In [ ]:
from allpower.scenarios import demand_history, scenario_output
from allpower.backup import plan
from allpower.scenarios import HOSPITAL
print(json.dumps(HOSPITAL, indent=2))
print(demand_history()[0], demand_history()[-1])


## 2. Compare the three scenarios

A regional deficit is a modeled tight-margin condition. A separate fictional local interruption begins at the first deficit hour to test hospital preparation.

In [ ]:
for name in ("normal", "supply-loss", "storm"):
    r=scenario_output(name)
    b=plan(r)
    print(name, {"first_deficit_hour":r["first_deficit_hour"], "deficit_mwh":r["deficit_energy_mwh"], "critical_served_kwh":b["critical_served_kwh"], "improvement_kwh":b["improvement_kwh"], "method":b["method"], "provenance":r["provenance"]})


## 3. Add observed hourly demand

Download ERCOT hourly load or EIA-930 from the linked official sites. Preserve the original file, URL, download time, timezone, and hash. Convert it to `data/ercot_hourly.csv` with `timestamp,demand_mw` in continuous hourly UTC. No missing hour may be silently filled.

In [ ]:
from scripts.train_demand import load
source=Path("data/ercot_hourly.csv")
if source.exists():
    rows=load(source)
    print(len(rows), rows[0], rows[-1])
else:
    print("Observed data not staged. The remaining evaluation is intentionally skipped.")


## 4. Train six direct candidates and inspect honest baselines

The script uses chronological train, validation, and test windows with six-hour separation. Candidate selection uses validation MAE only. The untouched test report includes MAE, RMSE and interval coverage by horizon. XGBoost is preferred when importable; scikit-learn histogram gradient boosting is the fallback.

In [ ]:
import subprocess
if source.exists():
    subprocess.run([sys.executable, "scripts/train_demand.py", str(source), "--out", "artifacts/demand"], check=True)
    report=json.loads(Path("artifacts/demand/evaluation.json").read_text())
    print(json.dumps(report, indent=2))
else:
    print("No real-data metrics have been generated.")


## 5. Vision work

The app currently has a simulated asset concern and a disabled image endpoint. After checking the InsPLAD license and source-image grouping, stage one category in `data/vision/{train,val,test}/{normal,defect}`. Run `python3 scripts/train_vision.py data/vision`. Its output compares HOG logistic regression and a compact CNN. Do not cite performance until the source-image group split is verified.

## Interpretation

Forecast demand is an input to a scenario margin calculation. Supply adequacy needs timestamped capacity data, while a hospital-level interruption requires local feeder evidence. The backup plan measures modeled critical kWh under disclosed assumptions. None of these outputs is an operational alert or clinical outcome.